# Thực hành Bài kiểm tra 1 – ViHSD thật

Notebook mặc định dùng ba split chính thức đã lưu tại `data/raw/uitnlp_vihsd/`. Nếu thư mục chưa có, notebook gọi `load_dataset("uitnlp/vihsd")` nên máy cần đăng nhập Hugging Face và đã được cấp quyền. Chế độ fixture chỉ bật thủ công để kiểm thử kỹ thuật; không dùng fixture làm số liệu nghiên cứu. Notebook chỉ in thống kê tổng hợp, không hiển thị bình luận.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset, load_from_disk

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vihsd.data import validate_schema
from vihsd.eda import comment_lengths, count_labels, length_summary, plot_comment_lengths, plot_label_distribution
from vihsd.preparation import fit_tfidf_on_train
from vihsd.preprocessing import normalize_series, preprocessing_summary
from vihsd.quality import cross_split_duplicate_summary, quality_issue_table

DATASET_ID = "uitnlp/vihsd"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "uitnlp_vihsd"
TEXT_COLUMN = "free_text"
LABEL_COLUMN = "label_id"
LABEL_NAMES = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}
USE_SAMPLE_FIXTURE = os.environ.get("VIHSD_NOTEBOOK_USE_FIXTURE") == "1"
OUTPUT_DIR = Path(os.environ.get("VIHSD_NOTEBOOK_OUTPUT_DIR", PROJECT_ROOT / "artifacts" / "practice"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if USE_SAMPLE_FIXTURE:
    frames = {
        "train": pd.DataFrame({TEXT_COLUMN: ["xin chào", "ko thích", "cảm ơn", "mẫu kỹ thuật"], LABEL_COLUMN: [0, 1, 0, 2]}),
        "validation": pd.DataFrame({TEXT_COLUMN: ["bình luận mẫu"], LABEL_COLUMN: [2]}),
        "test": pd.DataFrame({TEXT_COLUMN: ["ví dụ kỹ thuật"], LABEL_COLUMN: [0]}),
    }
    SOURCE_MODE = "FIXTURE MINH HOA — KHONG PHAI ViHSD"
else:
    if DATA_PATH.exists():
        dataset = load_from_disk(str(DATA_PATH))
    else:
        dataset = load_dataset(DATASET_ID)
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        dataset.save_to_disk(str(DATA_PATH))
    frames = {name: split.to_pandas() for name, split in dataset.items()}
    SOURCE_MODE = f"Dữ liệu thật: {DATASET_ID}"

for split_name, split_frame in frames.items():
    validate_schema(split_frame, TEXT_COLUMN, LABEL_COLUMN)
    print(f"{split_name}: {len(split_frame):,} dòng | {split_frame[TEXT_COLUMN].dtype} | {split_frame[LABEL_COLUMN].dtype}")
print(SOURCE_MODE)
print("Các giá trị nhãn:", sorted(set(pd.concat(frames.values())[LABEL_COLUMN].dropna().unique())))

## 1. Phân bố nhãn

Đọc số lượng và tỷ lệ theo từng split. Với fixture, chỉ dùng để thử mã.

In [ ]:
labels = []
for split_name, split_frame in frames.items():
    counts = count_labels(split_frame, LABEL_COLUMN)
    counts.insert(0, "split", split_name)
    counts["label_name"] = counts["label"].map(LABEL_NAMES)
    labels.append(counts)
label_table = pd.concat(labels, ignore_index=True)
print(label_table[["split", "label_name", "count", "percent"]].to_string(index=False))
pooled_counts = count_labels(pd.concat(frames.values(), ignore_index=True), LABEL_COLUMN)
label_counts = pooled_counts
pooled_counts["label"] = pooled_counts["label"].map(LABEL_NAMES)
figure_path = OUTPUT_DIR / "label_distribution.png"
plot_label_distribution(pooled_counts, figure_path)
plt.figure(figsize=(8, 5))
plt.imshow(plt.imread(figure_path))
plt.axis("off")
if not os.environ.get("VIHSD_NOTEBOOK_SUPPRESS_DISPLAY"):
    plt.show()

## 2. Độ dài bình luận

Độ dài ký tự và số đơn vị tách bởi khoảng trắng là đặc trưng mô tả. Số đơn vị cách trắng chưa phải số từ PyVi.

In [ ]:
all_frame = pd.concat(frames.values(), ignore_index=True)
all_frame = all_frame.copy()
all_frame["label_name"] = all_frame[LABEL_COLUMN].map(LABEL_NAMES)
all_frame_for_lengths = all_frame.rename(columns={LABEL_COLUMN: "label"}).copy()
all_frame_for_lengths["label"] = all_frame_for_lengths["label"].map(LABEL_NAMES)
lengths = comment_lengths(all_frame_for_lengths, TEXT_COLUMN, "label")
print(length_summary(lengths).to_string(index=False))
length_path = OUTPUT_DIR / "comment_lengths.png"
plot_comment_lengths(lengths, length_path)
plt.figure(figsize=(9, 5))
plt.imshow(plt.imread(length_path))
plt.axis("off")
if not os.environ.get("VIHSD_NOTEBOOK_SUPPRESS_DISPLAY"):
    plt.show()

## 3. Chất lượng dữ liệu và trùng giữa split

Các ví dụ văn bản được lược bỏ trong bảng chất lượng. Quy tắc dò trùng chuẩn hóa Unicode NFC, khoảng trắng và chữ hoa/thường; chưa tự xóa dòng.

In [ ]:
allowed_labels = list(LABEL_NAMES)
issues = quality_issue_table(all_frame, TEXT_COLUMN, LABEL_COLUMN, allowed_labels)
print(issues[["issue", "count", "percent_of_rows", "safe_example"]].to_string(index=False))
overlap = cross_split_duplicate_summary(frames, TEXT_COLUMN)
print(overlap.to_string(index=False))

## 4. Chuẩn hóa có kiểm soát

Chuẩn hóa Unicode, khoảng trắng và một danh sách viết tắt nhỏ trên bản dẫn xuất. So sánh chỉ báo tổng hợp; không in văn bản trước/sau và không đổi nhãn.

In [ ]:
processed_frames = {}
preprocessing_rows = []
for split_name, split_frame in frames.items():
    processed = normalize_series(split_frame[TEXT_COLUMN])
    processed_frames[split_name] = pd.DataFrame({
        TEXT_COLUMN: processed.fillna("").astype(str),
        LABEL_COLUMN: split_frame[LABEL_COLUMN].copy(),
    })
    preprocessing_rows.append({
        "split": split_name,
        **preprocessing_summary(split_frame[TEXT_COLUMN], processed),
        "missing_text_rows": int(split_frame[TEXT_COLUMN].isna().sum()),
    })
print(pd.DataFrame(preprocessing_rows).to_string(index=False))

## 5. Chuẩn bị TF-IDF

Giữ nguyên split chính thức. Vectorizer từ dùng PyVi với n-gram `(1, 2)`; vectorizer ký tự dùng n-gram `(2, 5)`. Chỉ `fit` trên Train, chỉ `transform` Validation/Test. Không huấn luyện bộ phân loại.

In [ ]:
train = processed_frames["train"]
validation = processed_frames["validation"]
test = processed_frames["test"]
features = fit_tfidf_on_train(
    train[TEXT_COLUMN].tolist(),
    validation[TEXT_COLUMN].tolist(),
    test[TEXT_COLUMN].tolist(),
    train[LABEL_COLUMN],
    validation[LABEL_COLUMN],
    test[LABEL_COLUMN],
)
feature_rows = []
for split_name in ("train", "validation", "test"):
    word = getattr(features, f"word_{split_name}")
    char = getattr(features, f"char_{split_name}")
    combined = getattr(features, f"combined_{split_name}")
    feature_rows.append({
        "split": split_name,
        "rows": combined.shape[0],
        "word_features": word.shape[1],
        "character_features": char.shape[1],
        "combined_features": combined.shape[1],
        "nnz": combined.nnz,
        "operation": "fit_transform" if split_name == "train" else "transform",
    })
print(pd.DataFrame(feature_rows).to_string(index=False))
print("Vocabulary chỉ được học từ Train:", len(features.word_vectorizer.vocabulary_), "n-gram từ và", len(features.char_vectorizer.vocabulary_), "n-gram ký tự")

## Câu hỏi ôn tập

1. Vì sao F1 Macro phù hợp khi CLEAN chiếm khoảng 83% Train còn OFFENSIVE dưới 7%?
2. Vì sao bình luận HATE có thể dài hơn trong thống kê nhưng độ dài không đủ để xác định nhãn?
3. Các vectorizer đã fit ở split nào? Vì sao Validation/Test chỉ được transform?
4. Có bao nhiêu văn bản trùng Train–Test? Vì sao cần ghi rủi ro này trước khi báo cáo kết quả mô hình?
5. Khi chuẩn hóa từ viết tắt, tại sao phải so sánh trước/sau và giữ dữ liệu/nhãn gốc?